# M5

In [1]:
import os
import sys
import warnings
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import wandb
from dotenv import load_dotenv
from transformers import AutoTokenizer, AutoModelForSequenceClassification

sys.path.append(os.path.abspath('..'))
from src.utils import map_at_3, load_config, set_seed

warnings.filterwarnings('ignore')
load_dotenv('../.env')
set_seed(42)

OPTION_COLS  = ['A', 'B', 'C', 'D', 'E']
LABEL_MAP    = {0: 'A', 1: 'B', 2: 'C', 3: 'D', 4: 'E'}
DEVICE       = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
MAX_LENGTH   = 256

train_df = pd.read_csv('../data/raw/train.csv')
test_df  = pd.read_csv('../data/raw/test.csv')

print(f'Device     : {DEVICE}')
print(f'Train shape: {train_df.shape}')
print(f'Test shape : {test_df.shape}')

/Users/risshabsrinivas/Documents/IITM/DL_GENAI_PROJECT/dl-genai-project-26-t2/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device     : mps
Train shape: (2000, 8)
Test shape : (500, 7)


In [2]:
DEBERTA_NAME = 'microsoft/deberta-v3-small'
ROBERTA_NAME = 'roberta-base'

print('DeBERTa')
deberta_tokenizer = AutoTokenizer.from_pretrained(DEBERTA_NAME)
deberta_model     = AutoModelForSequenceClassification.from_pretrained(
    DEBERTA_NAME, num_labels=5, ignore_mismatched_sizes=True
).to(DEVICE)
deberta_model.eval()

print('RoBERTa')
roberta_tokenizer = AutoTokenizer.from_pretrained(ROBERTA_NAME)
roberta_model     = AutoModelForSequenceClassification.from_pretrained(
    ROBERTA_NAME, num_labels=5, ignore_mismatched_sizes=True
).to(DEVICE)
roberta_model.eval()



DeBERTa


Loading weights: 100%|██████████| 102/102 [00:00<00:00, 34576.82it/s]
[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
pooler.dense.weight                     | MISSING    | 
classifier.bias                         | MISSING  

RoBERTa


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 7567.62it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.bias               | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
  

In [4]:
def build_input(row):
    prompt = str(row['prompt'])
    opts   = ' | '.join([f"{c}: {str(row[c])}" for c in OPTION_COLS])
    return prompt + ' ' + opts


def get_probs(model, tokenizer, text):
    enc = tokenizer(
        text,
        max_length=MAX_LENGTH,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    ).to(DEVICE)
    with torch.no_grad():
        logits = model(**enc).logits
    return F.softmax(logits, dim=-1)[0].cpu().numpy()


def probs_to_top3(probs):
    ranked = np.argsort(probs)[::-1]
    return [LABEL_MAP[i] for i in ranked[:3]]


def probs_to_top1(probs):
    return LABEL_MAP[int(np.argmax(probs))]




In [5]:
row25      = test_df.iloc[25]
text_25    = build_input(row25)

deberta_probs_25 = get_probs(deberta_model, deberta_tokenizer, text_25)
roberta_probs_25 = get_probs(roberta_model, roberta_tokenizer, text_25)

q1_option = probs_to_top1(deberta_probs_25)
q1_prob   = round(float(np.max(deberta_probs_25)), 4)

print(f'DeBERTa probs : { {LABEL_MAP[i]: round(float(p), 4) for i, p in enumerate(deberta_probs_25)} }')
print(f'ANS: {q1_option}, {q1_prob}')

DeBERTa probs : {'A': 0.2151, 'B': 0.1964, 'C': 0.224, 'D': 0.1366, 'E': 0.2279}
ANS: E, 0.2279


In [6]:
avg_probs_25 = (deberta_probs_25 + roberta_probs_25) / 2
q2_answer    = probs_to_top1(avg_probs_25)

print(f'RoBERTa probs : { {LABEL_MAP[i]: round(float(p), 4) for i, p in enumerate(roberta_probs_25)} }')
print(f'Avg probs     : { {LABEL_MAP[i]: round(float(p), 4) for i, p in enumerate(avg_probs_25)} }')
print(f'ANS: {q2_answer}')

RoBERTa probs : {'A': 0.1947, 'B': 0.2385, 'C': 0.1921, 'D': 0.217, 'E': 0.1577}
Avg probs     : {'A': 0.2049, 'B': 0.2174, 'C': 0.2081, 'D': 0.1768, 'E': 0.1928}
ANS: B


In [7]:
W_DEBERTA = 0.70
W_ROBERTA = 0.30

weighted_probs_25 = (W_DEBERTA * deberta_probs_25) + (W_ROBERTA * roberta_probs_25)
q3_answer         = probs_to_top1(weighted_probs_25)

print(f'Weighted probs: { {LABEL_MAP[i]: round(float(p), 4) for i, p in enumerate(weighted_probs_25)} }')
print(f'ANS: {q3_answer}')

Weighted probs: {'A': 0.209, 'B': 0.2091, 'C': 0.2145, 'D': 0.1608, 'E': 0.2069}
ANS: C


In [8]:
top3_25   = probs_to_top3(weighted_probs_25)
q4_answer = ' '.join(top3_25)

print(f'ANS: {q4_answer}')

ANS: C B A


In [9]:
results = []

for idx, row in test_df.iterrows():
    text         = build_input(row)
    deb_probs    = get_probs(deberta_model, deberta_tokenizer, text)
    rob_probs    = get_probs(roberta_model, roberta_tokenizer, text)
    w_probs      = (W_DEBERTA * deb_probs) + (W_ROBERTA * rob_probs)
    top3         = probs_to_top3(w_probs)
    results.append({'id': row['id'], 'prediction': ' '.join(top3)})

    if (idx + 1) % 50 == 0:
        print(f'Processed {idx + 1} / {len(test_df)}')

submission    = pd.DataFrame(results)
submission.to_csv('../submission.csv', index=False)
q5_answer     = len(submission)


print(submission.head())
print(f'(prediction rows): {q5_answer}')

Processed 50 / 500
Processed 100 / 500
Processed 150 / 500
Processed 200 / 500
Processed 250 / 500
Processed 300 / 500
Processed 350 / 500
Processed 400 / 500
Processed 450 / 500
Processed 500 / 500
   id prediction
0   1      C B A
1   2      C B A
2   3      C B A
3   4      C B A
4   5      C B A
(prediction rows): 500


In [10]:
TTA_PREFIX   = 'Answer the following multiple-choice question carefully: '
tta_changes  = 0

for i in range(50):
    row          = test_df.iloc[i]
    text_orig    = build_input(row)
    text_aug     = TTA_PREFIX + text_orig

    probs_orig   = get_probs(deberta_model, deberta_tokenizer, text_orig)
    probs_aug    = get_probs(deberta_model, deberta_tokenizer, text_aug)
    probs_tta    = (probs_orig + probs_aug) / 2

    top1_orig    = probs_to_top1(probs_orig)
    top1_tta     = probs_to_top1(probs_tta)

    if top1_orig != top1_tta:
        tta_changes += 1
        print(f'  Row {i:3d}: {top1_orig} → {top1_tta}')

q6_answer = tta_changes
print(f'ANS: {q6_answer}')

ANS: 0


In [11]:
disagreements = 0

deb_top1_100  = []
ens_top1_100  = []
deb_conf_100  = []
ens_conf_100  = []
deb_top3_100  = []
ens_top3_100  = []

for i in range(100):
    row       = test_df.iloc[i]
    text      = build_input(row)

    deb_probs = get_probs(deberta_model, deberta_tokenizer, text)
    rob_probs = get_probs(roberta_model, roberta_tokenizer, text)
    w_probs   = (W_DEBERTA * deb_probs) + (W_ROBERTA * rob_probs)

    d_top1    = probs_to_top1(deb_probs)
    e_top1    = probs_to_top1(w_probs)
    d_top3    = probs_to_top3(deb_probs)
    e_top3    = probs_to_top3(w_probs)

    deb_top1_100.append(d_top1)
    ens_top1_100.append(e_top1)
    deb_conf_100.append(float(np.max(deb_probs)))
    ens_conf_100.append(float(np.max(w_probs)))
    deb_top3_100.append(d_top3)
    ens_top3_100.append(e_top3)

    if d_top1 != e_top1:
        disagreements += 1

    if (i + 1) % 25 == 0:
        print(f'Processed {i + 1} / 100')

q7_answer = disagreements
print(f'ANS: {q7_answer}')

Processed 25 / 100
Processed 50 / 100
Processed 75 / 100
Processed 100 / 100
ANS: 96


In [12]:
conf_gains       = [ens_conf_100[i] - deb_conf_100[i] for i in range(100)]
positive_gains   = sum(1 for g in conf_gains if g > 0)
q8_answer        = positive_gains

print(f'Mean confidence gain : {round(np.mean(conf_gains), 4)}')
print(f'(rows with positive confidence gain): {q8_answer}')

Mean confidence gain : -0.0117
(rows with positive confidence gain): 0


In [13]:
top3_changes = 0

for i in range(100):
    if deb_top3_100[i] != ens_top3_100[i]:
        top3_changes += 1

q9_answer = top3_changes
print(f'(rows with at least one Top-3 change): {q9_answer}')

(rows with at least one Top-3 change): 100


In [14]:
val_preds  = []
val_labels = []

for i in range(100):
    row       = train_df.iloc[i]
    text      = build_input(row)

    deb_probs = get_probs(deberta_model, deberta_tokenizer, text)
    rob_probs = get_probs(roberta_model, roberta_tokenizer, text)
    w_probs   = (W_DEBERTA * deb_probs) + (W_ROBERTA * rob_probs)

    val_preds.append(probs_to_top3(w_probs))
    val_labels.append(row['answer'])

    if (i + 1) % 25 == 0:
        print(f'Processed {i + 1} / 100')

q10_answer = round(map_at_3(val_preds, val_labels), 4)
print(f'(MAP@3 on train first 100): {q10_answer}')

Processed 25 / 100
Processed 50 / 100
Processed 75 / 100
Processed 100 / 100
(MAP@3 on train first 100): 0.435


## Consolidated Answers

In [15]:

print(f'  Q1   DeBERTa Top-1 (row 25)                  : {q1_option}, {q1_prob}')
print(f'  Q2   Simple avg ensemble Top-1 (row 25)      : {q2_answer}')
print(f'  Q3   Weighted ensemble Top-1 (row 25)        : {q3_answer}')
print(f'  Q4   Top-3 prediction string (row 25)        : {q4_answer}')
print(f'  Q5   Rows in submission.csv                  : {q5_answer}')
print(f'  Q6   TTA Top-1 changes (first 50)            : {q6_answer}')
print(f'  Q7   DeBERTa vs ensemble Top-1 diff (100)    : {q7_answer}')
print(f'  Q8   Rows with positive confidence gain      : {q8_answer}')
print(f'  Q9   Rows with Top-3 ranking change          : {q9_answer}')
print(f'  Q10  Weighted ensemble MAP@3 (train 100)     : {q10_answer}')


  Q1   DeBERTa Top-1 (row 25)                  : E, 0.2279
  Q2   Simple avg ensemble Top-1 (row 25)      : B
  Q3   Weighted ensemble Top-1 (row 25)        : C
  Q4   Top-3 prediction string (row 25)        : C B A
  Q5   Rows in submission.csv                  : 500
  Q6   TTA Top-1 changes (first 50)            : 0
  Q7   DeBERTa vs ensemble Top-1 diff (100)    : 96
  Q8   Rows with positive confidence gain      : 0
  Q9   Rows with Top-3 ranking change          : 100
  Q10  Weighted ensemble MAP@3 (train 100)     : 0.435
